<a href="https://colab.research.google.com/github/athishsreeram/tubenotebook/blob/main/TubeNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📺 TubeNotebook
### YouTube Channel Intelligence Tool

**What this does:**
- ✅ Fetch all video titles + URLs from any YouTube channel
- ✅ Deep-fetch metadata (views, likes, comments, duration, tags)
- ✅ Pull full transcripts for any video
- ✅ Compute engagement scores
- ✅ Export everything to CSV + JSON

**No API key needed. Works in Google Colab.**

---

## Step 1 — Install Dependencies

In [1]:
# Run this first — installs both required packages
!pip install yt-dlp youtube-transcript-api --quiet
print("✅ Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 29.3 MB/s eta 0:00:00
✅ Dependencies installed.


## Step 2 — Configuration
**Edit this cell to target any channel.**

In [2]:
# ─────────────────────────────────────────
#  ✏️  EDIT THESE — everything else is auto
# ─────────────────────────────────────────

CHANNEL_URL      = "https://www.youtube.com/@rajshamani/videos"  # Any channel URL
MAX_VIDEOS       = 50      # Max videos to scan (None = all)
DEEP_FETCH_TOP_N = 5       # How many top videos to deep-fetch (metadata + transcript)
FETCH_TRANSCRIPTS = True   # Set False to skip transcripts (faster)
TRANSCRIPT_LANG  = "en"    # Language code: "en", "hi", "ta", etc.

print(f"🎯 Target channel : {CHANNEL_URL}")
print(f"📊 Max videos     : {MAX_VIDEOS or 'All'}")
print(f"🔬 Deep-fetch top : {DEEP_FETCH_TOP_N} videos")
print(f"📝 Transcripts    : {'Yes' if FETCH_TRANSCRIPTS else 'No'}")

🎯 Target channel : https://www.youtube.com/@rajshamani/videos
📊 Max videos     : 50
🔬 Deep-fetch top : 5 videos
📝 Transcripts    : Yes


## Step 3 — Imports & Helpers

In [3]:
import yt_dlp
import json
import csv
import time
from datetime import datetime
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

# ── Helpers ──────────────────────────────

def format_duration(seconds):
    """Convert seconds → MM:SS or HH:MM:SS"""
    if not seconds:
        return "N/A"
    s = int(seconds)
    h, m, s = s // 3600, (s % 3600) // 60, s % 60
    return f"{h}:{m:02}:{s:02}" if h else f"{m}:{s:02}"

def fmt_number(n):
    """Format large numbers with commas"""
    return f"{n:,}" if n else "N/A"

def engagement_score(likes, comments, views):
    """(likes + comments) / views — how interactive a video is"""
    if not views or views == 0:
        return 0.0
    return round(((likes or 0) + (comments or 0)) / views * 100, 4)

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

print("✅ Imports ready.")

✅ Imports ready.


## Step 4 — Fetch All Video URLs from Channel

In [4]:
def fetch_channel_videos(channel_url: str, max_videos: int = None) -> list[dict]:
    """
    Fast-fetch all video titles + URLs from a channel.
    Uses extract_flat=True — no video download, just metadata index.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "extract_flat": True,
        "skip_download": True,
        "ignoreerrors": True,
        "playlistend": max_videos,
    }

    videos = []
    print(f"\n🔍 Scanning channel: {channel_url}")
    print("⏳ Fetching video list...\n")

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)

        if not info or "entries" not in info:
            print("❌ No videos found. Check the URL.")
            return []

        channel_name = info.get("uploader") or info.get("channel") or "Unknown"

        for entry in info["entries"]:
            if not entry:
                continue
            vid_id = entry.get("id", "")
            url    = entry.get("url") or f"https://www.youtube.com/watch?v={vid_id}"
            if not url.startswith("http"):
                url = f"https://www.youtube.com/watch?v={vid_id}"

            videos.append({
                "video_id":    vid_id,
                "title":       entry.get("title", "Untitled"),
                "url":         url,
                "channel":     channel_name,
                "upload_date": entry.get("upload_date"),
                "duration":    entry.get("duration"),
                "view_count":  entry.get("view_count"),
            })

    return videos


# ── Run it ───────────────────────────────
all_videos = fetch_channel_videos(CHANNEL_URL, max_videos=MAX_VIDEOS)

print(f"{'─'*70}")
print(f"{'#':<5} {'TITLE':<52} {'DURATION'}")
print(f"{'─'*70}")
for i, v in enumerate(all_videos, 1):
    title = v['title'][:49] + '...' if len(v['title']) > 52 else v['title']
    print(f"{i:<5} {title:<52} {format_duration(v['duration'])}")
print(f"{'─'*70}")
print(f"\n✅ Found {len(all_videos)} videos.")


🔍 Scanning channel: https://www.youtube.com/@rajshamani/videos
⏳ Fetching video list...

──────────────────────────────────────────────────────────────────────
#     TITLE                                                DURATION
──────────────────────────────────────────────────────────────────────
1     Khan Sir on World War 3, India vs Pakistan, China... 1:38:21
2     The Invisible Nuclear Bomb - Strait of Hormuz & G... 2:21:26
3     ⁠Dawood vs Rehman Dakait: Karachi Underworld, ISI... 1:31:23
4     Why USA-Israel Is Attacking Iran: War Breakdown, ... 2:08:28
5     Are Successful People Broken? Dark Traits, AI Evo... 1:47:41
6     AI Masterclass: Become an Expert at Claude, Gemin... 2:18:56
7     Reverse Ageing in Parents: Social Life, Brain Hea... 1:33:08
8     Shreya Ghoshal on Reality Shows, Stardom, Lip Syn... 2:20:37
9     Suniel Shetty on Bollywood’s Reality, Father’s De... 1:33:36
10    Simon Sinek on India’s Future, Trust Crisis, Gen ... 1:36:40
11    Dr Joe Dispenza: Rewire 

## Step 5 — Deep Fetch: Metadata for Top N Videos
This fetches rich metadata (views, likes, comments, tags) for the top N videos.

In [5]:
def deep_fetch_video(url: str) -> dict:
    """
    Full metadata fetch for a single video — views, likes, comments, tags, description.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "ignoreerrors": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    if not info:
        return {}

    views    = info.get("view_count") or 0
    likes    = info.get("like_count") or 0
    comments = info.get("comment_count") or 0

    return {
        "video_id":     info.get("id"),
        "title":        info.get("title"),
        "url":          f"https://www.youtube.com/watch?v={info.get('id')}",
        "channel":      info.get("uploader") or info.get("channel"),
        "upload_date":  info.get("upload_date"),
        "duration_sec": info.get("duration"),
        "duration_fmt": format_duration(info.get("duration")),
        "view_count":   views,
        "like_count":   likes,
        "comment_count":comments,
        "engagement_%": engagement_score(likes, comments, views),
        "tags":         ", ".join(info.get("tags") or []),
        "description":  (info.get("description") or "")[:500],  # first 500 chars
        "thumbnail":    info.get("thumbnail"),
    }


# ── Run deep fetch on top N ───────────────
top_videos = all_videos[:DEEP_FETCH_TOP_N]
rich_data  = []

print(f"\n🔬 Deep-fetching top {len(top_videos)} videos...\n")

for i, v in enumerate(top_videos, 1):
    print(f"  [{i}/{len(top_videos)}] {v['title'][:60]}")
    data = deep_fetch_video(v["url"])
    if data:
        rich_data.append(data)
    time.sleep(0.5)  # be polite

print(f"\n✅ Deep-fetched {len(rich_data)} videos.\n")

# Pretty print summary table
print(f"{'─'*90}")
print(f"{'TITLE':<40} {'VIEWS':>10} {'LIKES':>8} {'COMMENTS':>9} {'ENGAGE%':>8} {'DURATION':>9}")
print(f"{'─'*90}")
for v in rich_data:
    t = v['title'][:37] + '...' if len(v['title']) > 40 else v['title']
    print(f"{t:<40} {fmt_number(v['view_count']):>10} {fmt_number(v['like_count']):>8} "
          f"{fmt_number(v['comment_count']):>9} {v['engagement_%']:>8} {v['duration_fmt']:>9}")
print(f"{'─'*90}")


🔬 Deep-fetching top 5 videos...

  [1/5] Khan Sir on World War 3, India vs Pakistan, China, Trump & E
  [2/5] The Invisible Nuclear Bomb - Strait of Hormuz & Global Econo
  [3/5] ⁠Dawood vs Rehman Dakait: Karachi Underworld, ISI & Drug Tra
  [4/5] Why USA-Israel Is Attacking Iran: War Breakdown, Dubai & Tru
  [5/5] Are Successful People Broken? Dark Traits, AI Evolution & Lo

✅ Deep-fetched 5 videos.

──────────────────────────────────────────────────────────────────────────────────────────
TITLE                                         VIEWS    LIKES  COMMENTS  ENGAGE%  DURATION
──────────────────────────────────────────────────────────────────────────────────────────
Khan Sir on World War 3, India vs Pak...  3,704,079  152,705     8,700   4.3575   1:38:21
The Invisible Nuclear Bomb - Strait o...  1,506,362   25,974     2,400   1.8836   2:21:25
⁠Dawood vs Rehman Dakait: Karachi Und...  2,284,215   34,895     1,400   1.5889   1:31:23
Why USA-Israel Is Attacking Iran: War...  1,334,408 

## Step 6 — Fetch Transcripts

In [6]:
def get_transcript(video_id: str, lang: str = "en") -> str:
    """
    Fetch full transcript for a video.
    Falls back to auto-generated captions if manual captions unavailable.
    Returns plain text string.
    """
    try:
        # Try requested language first
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
    except NoTranscriptFound:
        try:
            # Fall back to any available language
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            transcript = transcript_list.find_generated_transcript(
                transcript_list._generated_transcripts.keys()
            ).fetch()
        except Exception:
            return "[No transcript available]"
    except TranscriptsDisabled:
        return "[Transcripts disabled for this video]"
    except Exception as e:
        return f"[Error: {str(e)}]"

    return " ".join(seg["text"] for seg in transcript)


# ── Attach transcripts to rich data ──────
if FETCH_TRANSCRIPTS and rich_data:
    print(f"\n📝 Fetching transcripts for {len(rich_data)} videos...\n")
    for i, v in enumerate(rich_data, 1):
        print(f"  [{i}/{len(rich_data)}] {v['title'][:60]}")
        transcript_text = get_transcript(v["video_id"], lang=TRANSCRIPT_LANG)
        v["transcript"]          = transcript_text
        v["transcript_word_count"] = len(transcript_text.split())
        time.sleep(0.3)

    print(f"\n✅ Transcripts attached.")

    # Preview first transcript
    if rich_data:
        first = rich_data[0]
        print(f"\n📄 Transcript preview — {first['title']}")
        print(f"   Word count: {first['transcript_word_count']:,}")
        print(f"   Preview: {first['transcript'][:400]}...")
else:
    print("⏭️  Transcript fetching skipped (FETCH_TRANSCRIPTS=False)")


📝 Fetching transcripts for 5 videos...

  [1/5] Khan Sir on World War 3, India vs Pakistan, China, Trump & E
  [2/5] The Invisible Nuclear Bomb - Strait of Hormuz & Global Econo
  [3/5] ⁠Dawood vs Rehman Dakait: Karachi Underworld, ISI & Drug Tra
  [4/5] Why USA-Israel Is Attacking Iran: War Breakdown, Dubai & Tru
  [5/5] Are Successful People Broken? Dark Traits, AI Evolution & Lo

✅ Transcripts attached.

📄 Transcript preview — Khan Sir on World War 3, India vs Pakistan, China, Trump & Epstein Files | FO485 Raj Shamani
   Word count: 8
   Preview: [Error: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript']...


## Step 7 — Export to CSV & JSON

In [7]:
ts = timestamp()

# ── Export 1: Full channel video list (title + URL only) ──
csv_url_file = f"tube_videos_{ts}.csv"
with open(csv_url_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "url", "video_id", "upload_date",
                                           "duration", "view_count", "channel"])
    writer.writeheader()
    writer.writerows(all_videos)
print(f"💾 All videos (URL list)   → {csv_url_file}  ({len(all_videos)} rows)")


# ── Export 2: Rich metadata CSV ──
if rich_data:
    csv_rich_file = f"tube_rich_{ts}.csv"
    rich_keys = [k for k in rich_data[0].keys() if k != "transcript"]  # skip transcript in CSV
    with open(csv_rich_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rich_keys, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rich_data)
    print(f"💾 Rich metadata CSV       → {csv_rich_file}  ({len(rich_data)} rows)")


# ── Export 3: Full JSON (includes transcripts) ──
if rich_data:
    json_file = f"tube_full_{ts}.json"
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(rich_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Full JSON (+ transcripts) → {json_file}  ({len(rich_data)} videos)")


print("\n✅ All exports done. Download from the Files panel (left sidebar in Colab).")

💾 All videos (URL list)   → tube_videos_20260321_030427.csv  (50 rows)
💾 Rich metadata CSV       → tube_rich_20260321_030427.csv  (5 rows)
💾 Full JSON (+ transcripts) → tube_full_20260321_030427.json  (5 videos)

✅ All exports done. Download from the Files panel (left sidebar in Colab).


## Step 8 — Bonus: Get Transcript for Any Single Video

In [8]:
# ✏️  Paste any YouTube video URL here
SINGLE_VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

# Extract video ID
video_id_single = SINGLE_VIDEO_URL.split("v=")[-1].split("&")[0]

print(f"📄 Fetching transcript for: {SINGLE_VIDEO_URL}")
text = get_transcript(video_id_single, lang=TRANSCRIPT_LANG)

word_count = len(text.split())
print(f"\n✅ Word count: {word_count:,}")
print(f"\n--- Transcript (first 800 chars) ---")
print(text[:800])
print("...")

# Save to file
out_file = f"transcript_{video_id_single}.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(text)
print(f"\n💾 Full transcript saved to: {out_file}")

📄 Fetching transcript for: https://www.youtube.com/watch?v=dQw4w9WgXcQ

✅ Word count: 8

--- Transcript (first 800 chars) ---
[Error: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript']
...

💾 Full transcript saved to: transcript_dQw4w9WgXcQ.txt


---
**TubeNotebook** · Built by [@athishsreeram](https://github.com/athishsreeram) · [GitHub Repo](https://github.com/athishsreeram/tubenotebook)